# Hello World: Quantum Reservoir Classification with MerLin

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Quandela/Tutorials/blob/main/QML_with_MerLin/ReservoirClassifierCloudTutorial.ipynb)

In this tutorial, you will train a small image classifier and use a real photonic quantum processor to create its most important features. Run the cells from top to bottom; the only value you must provide is your Quandela Cloud token.

You will see what happens to an image at every stage, submit work to a QPU, train a small classical readout, and use the finished hybrid model to classify unseen handwritten digits.

## The experiment at a glance

`ReservoirClassifier` is a **hybrid quantum algorithm** with two classical parts and one quantum part:

| Stage | Where it runs | What happens | Is it trained? |
|---|---|---|---|
| **1. PCA** | Classical computer | Compresses 784 image pixels into a small list of values | Fitted once to the training images |
| **2. Photonic reservoir** | QPU | Encodes those values as optical phases, sends photons through a fixed interferometer, and measures output patterns | **No—the circuit stays fixed** |
| **3. Linear readout** | Classical computer | Learns which measured patterns indicate a 0 or a 1 | **Yes—these are the trainable weights** |

In short: **image → classical compression → quantum measurement → classical prediction**.

## 1. Install and import the dependencies

The next two cells prepare Google Colab or a local Jupyter environment. If prompted after a first installation, restart the kernel and continue from the top.

In [ ]:
### Run this cell to install the merlinquantum package
!pip install merlinquantum

### Check Perceval and MerLin Version

Let's check which version of Perceval and MerLin is installed to ensure compatibility.

In [ ]:
# Supporting code: imports packages and applies the visual style. You can just run this cell.
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import perceval as pcvl
import torch
import torch.nn as nn
import merlin as ML
from matplotlib.colors import LinearSegmentedColormap
from merlin import MerlinProcessor
from merlin.datasets import mnist_digits
from merlin.models import ReservoirClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import ConfusionMatrixDisplay
from torch.utils.data import DataLoader
from perceval import RemoteConfig

PRIMARY = "#435BEC"     # Indigo: inputs and transformations
ACCENT = "#F07362"      # Coral: predictions and key outcomes
BACKGROUND = "#000000"  # Black: figure backgrounds
FOREGROUND = "#FFFFFF"  # White: labels and outlines

plt.rcParams.update({
    "figure.facecolor": BACKGROUND,
    "axes.facecolor": BACKGROUND,
    "axes.edgecolor": FOREGROUND,
    "axes.labelcolor": FOREGROUND,
    "text.color": FOREGROUND,
    "xtick.color": FOREGROUND,
    "ytick.color": FOREGROUND,
})
RESULT_CMAP = LinearSegmentedColormap.from_list(
    "quandela_results", [BACKGROUND, PRIMARY, ACCENT]
)

print(f"MerLin version: {ML.__version__}")
print(f"Perceval version: {pcvl.__version__}")

## 2. Choose your experiment

Add your token, then run the supplied defaults once. Later, return to this cell and change one group at a time; no other code needs editing. Get a token from [Quandela Cloud](https://cloud.quandela.com), paste it below, and keep the quotation marks. Never commit or publish a notebook containing a real token.

The two available QPUs have different limits:

| Platform | Maximum photons | Maximum modes |
|---|---:|---:|
| `qpu:ascella` | 6 | 12 |
| `qpu:belenos` | 12 | 24 |

The settings are organised by purpose:

- **Data size:** more training images provide more examples to learn from; more test images make the final accuracy more informative. Both require more QPU evaluations.
- **Quantum circuit:** the platform sets the hardware limits. More PCA features retain more image information and require more modes (`PCA_FEATURES + 1`). More photons enlarge the measurement space and make the experiment more demanding.

In [ ]:
# ACCESS — this is the only value you must change.
CLOUD_TOKEN = "YOUR_CLOUD_TOKEN"  # Replace with your Quandela Cloud token

# DATA SIZE — larger values use the QPU more often.
TRAIN_IMAGES_PER_CLASS = 6  # Examples used to teach the readout.
TEST_IMAGES_PER_CLASS = 2   # Unseen examples used for the final check.

# QUANTUM CIRCUIT — these values must fit the chosen QPU.
CLOUD_PLATFORM = "qpu:ascella"  # Or "qpu:belenos".
PCA_FEATURES = 11  # Image values encoded as phases; the circuit uses one extra mode.
N_PHOTONS = 3     # photons sent through the reservoir for each image.

# Compact checks for the settings above; no changes are needed below this line.
QPU_LIMITS = {
    "qpu:ascella": {"photons": 6, "modes": 12},
    "qpu:belenos": {"photons": 12, "modes": 24},
}

print("Configuration ready:")
print(f"  platform: {CLOUD_PLATFORM} ({QPU_LIMITS[CLOUD_PLATFORM]['photons']} photons, {QPU_LIMITS[CLOUD_PLATFORM]['modes']} modes maximum)")
print(f"  dataset:  {2 * TRAIN_IMAGES_PER_CLASS} training and {2 * TEST_IMAGES_PER_CLASS} test images")
print(f"  circuit:  {N_PHOTONS} photons in {PCA_FEATURES + 1} modes")

## 3. Load the handwritten digits

MNIST contains 28 × 28 pixel images of handwritten digits. We will distinguish **0** from **1**. Pixel brightness is scaled from integers in `[0, 255]` to values in `[0, 1]`. A fixed random seed selects the same balanced examples on every run.

In [ ]:
# Supporting code: loads a balanced MNIST subset and defines a display helper. You can just run this cell.
CLASSES = (0, 1)
np.random.seed(123)
torch.manual_seed(123)

def load_mnist(split_function):
    images, labels, _ = split_function()
    images = np.asarray(images).reshape(len(images), -1).astype(np.float32) / 255.0
    return images, np.asarray(labels, dtype=np.int64)

def choose_examples(images, labels, per_class, seed):
    rng = np.random.default_rng(seed)
    selected = []
    for digit in CLASSES:
        available = np.flatnonzero(labels == digit)
        if per_class > len(available):
            raise ValueError(f"Requested {per_class} examples of {digit}, but only {len(available)} are available.")
        selected.extend(rng.choice(available, per_class, replace=False))
    rng.shuffle(selected)
    selected = np.asarray(selected)
    class_indices = np.array([CLASSES.index(int(labels[i])) for i in selected])
    return images[selected], class_indices

def show_digit_grid(images, labels, title, max_images=12):
    count = min(len(images), max_images)
    columns = min(6, count)
    rows = math.ceil(count / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(1.6 * columns, 1.8 * rows), squeeze=False)
    for ax, image, label in zip(axes.flat, images[:count], labels[:count]):
        ax.imshow(image.reshape(28, 28), cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"digit {CLASSES[int(label)]}", color=FOREGROUND)
        ax.axis("off")
    for ax in axes.flat[count:]:
        ax.axis("off")
    fig.suptitle(title, color=FOREGROUND)
    fig.tight_layout()
    plt.show()

X_train, y_train = choose_examples(
    *load_mnist(mnist_digits.get_data_train_original), TRAIN_IMAGES_PER_CLASS, 123
)
X_test, y_test = choose_examples(
    *load_mnist(mnist_digits.get_data_test_original), TEST_IMAGES_PER_CLASS, 124
)

print(f"Training set: {len(X_train)} images, {X_train.shape[1]} pixels each")
print(f"Test set:     {len(X_test)} images")

In [ ]:
# Supporting code: displays a readable selection of training images. You can just run this cell.
show_digit_grid(
    X_train, y_train,
    title=f"A selection from the {len(X_train)} training images",
    max_images=12,
)

## 4. Build the hybrid reservoir classifier

A raw image contains 784 values. PCA reduces it to `PCA_FEATURES` values on the classical computer. After scaling to `[0, 1]`, MerLin maps them to phase shifts between 0 and π in the optical circuit. One additional optical mode is used, so the circuit contains `PCA_FEATURES + 1` modes.

The displayed circuit reads from left to right: **fixed interferometer → phase shifters → fixed interferometer**. Look for the components labelled `PS phi=px1`, `px2`, and so on. Those phase shifters are exactly where the PCA features are embedded. Their settings change for each image; both interferometers remain fixed and are **not trained**.

After the photons pass through the circuit, the QPU measures their output patterns. The resulting probability distribution is the reservoir representation passed to the classical readout.

### Why are there so many possible outcomes?

For `n` indistinguishable photons in `m` optical modes, the full Fock-space dimension is

$$\dim \mathcal{F}_{m,n} = \binom{m+n-1}{n}. $$

This counts every way to distribute the photons, including several photons occupying one mode. `ReservoirClassifier` uses the collision-free (unbunched) outcomes, where each mode contains at most one photon. Its measured feature count is therefore

$$N_{\mathrm{outcomes}} = \binom{m}{n}. $$

In [ ]:
# Build the hybrid model: classical PCA → quantum reservoir → classical readout.
classifier = ReservoirClassifier(
    in_features=28 * 28,               # The 784 pixels entering PCA.
    out_features=len(CLASSES),         # One final score for 0 and one for 1.
    n_photons=N_PHOTONS,               # Photons prepared for each circuit run.
    reduction=PCA(n_components=PCA_FEATURES, random_state=123),
    concatenate=False,                 # The readout uses only quantum features.
    seed=123,                          # Fixes the random interferometers.
)

# The phase shifters px1...pxN in this circuit encode the PCA features.
# The two surrounding interferometers are fixed for the whole experiment.
modes = classifier.layer.n_modes
full_fock_dimension = math.comb(modes + N_PHOTONS - 1, N_PHOTONS)
collision_free_outcomes = math.comb(modes, N_PHOTONS)

print(f"Circuit: {N_PHOTONS} photons in {modes} modes")
print(f"Full Fock space: C({modes + N_PHOTONS - 1}, {N_PHOTONS}) = {full_fock_dimension} states")
print(f"Collision-free outcomes: C({modes}, {N_PHOTONS}) = {collision_free_outcomes}")
print(f"MerLin reservoir outputs: {len(classifier.layer.output_keys)} features")

# Display the circuit so the feature-encoding phase shifters are visible.
assert len(classifier.layer.output_keys) == collision_free_outcomes
pcvl.pdisplay(classifier.layer.circuit, recursive=False, render_size=0.6)

## 5. Connect MerLin to the QPU

`RemoteProcessor` connects Perceval to the selected cloud platform. `MerlinProcessor` translates MerLin inputs into QPU jobs and returns their measured probability distributions.

Each image produces one set of phase settings. The QPU repeats that circuit setting up to 2,000 times to estimate its output probabilities. A job may wait in the cloud queue; keep the notebook running while the cell waits for completion.

In [ ]:
# Connect Perceval to the physical QPU selected above.
remote_processor = pcvl.RemoteProcessor(CLOUD_PLATFORM, token=CLOUD_TOKEN)

# Wrap that connection so ReservoirClassifier can send image batches to it.
qpu = MerlinProcessor(
    processor=remote_processor,
    token=CLOUD_TOKEN,
    microbatch_size=min(4, len(X_train)),
    max_shots_per_call=2_000,  # Repetitions used to estimate each probability distribution.
    timeout=7_200.0,
)

print(f"Connected to: {remote_processor.name}")
print(f"Platform status: {remote_processor.status}")
print(f"Available commands: {sorted(qpu.available_commands)}")

## 6. Turn training images into QPU features

`fit_reservoir` fits the classical PCA, scales its values, embeds them in the circuit's phase shifters, and runs every training image through the fixed reservoir on the QPU. The measured output probability distribution becomes that image's quantum feature vector.

MerLin caches those QPU results. `make_dataset` therefore reuses them instead of submitting the training images again. The result is an ordinary PyTorch dataset: QPU-derived features paired with the known digit labels.

In [ ]:
# Time the QPU work so its cost is visible to the learner.
start = time.perf_counter()

# Fit PCA locally, encode every training image in the phase shifters,
# and measure the fixed circuit on the QPU. No circuit weights are trained.
classifier.fit_reservoir(X_train, processor=qpu)

# Reuse the cached QPU measurements and pair them with the known labels.
training_dataset = classifier.make_dataset(X_train, y_train, processor=qpu)
reservoir_features, training_labels = training_dataset.tensors

elapsed = time.perf_counter() - start
print(f"QPU feature extraction finished in {elapsed / 60:.1f} minutes")
print(f"Feature matrix: {len(X_train)} images × {reservoir_features.shape[1]} measured outcomes")

## 7. Train only the classical readout

The QPU work is complete and cached. This standard PyTorch loop runs locally and updates only the linear readout returned by `classifier.parameters()`. The fitted PCA and both quantum interferometers remain unchanged.

Cross-entropy loss measures how far the predictions are from the known labels. Adam adjusts the readout weights to reduce that loss.

In [ ]:
# Feed the cached QPU features to the classical readout in small batches.
loader = DataLoader(
    training_dataset,
    batch_size=min(8, len(training_dataset)),
    shuffle=True,
    generator=torch.Generator().manual_seed(123),
)
optimizer = torch.optim.Adam(classifier.parameters(), lr=0.02)
loss_function = nn.CrossEntropyLoss()

# Only classifier.parameters() are given to the optimizer. For this model,
# that means the final classical readout—not PCA and not the quantum circuit.
for _ in range(100):
    classifier.train()
    for features, targets in loader:
        optimizer.zero_grad()
        loss = loss_function(classifier(features), targets)
        loss.backward()
        optimizer.step()
classifier.eval()
with torch.no_grad():
    training_predictions = classifier(reservoir_features).argmax(dim=1)
    training_accuracy = (training_predictions == training_labels).float().mean().item()

print(f"Trainable readout parameters: {sum(p.numel() for p in classifier.parameters())}")
print("Frozen quantum parameters: all of them")
print(f"Training accuracy after 100 local epochs: {100 * training_accuracy:.1f}%")

## 8. Follow one image from pixels to classification

The four panels below show the complete hybrid path for one training image: the original pixels, the PCA values embedded as optical phases, the probability distribution measured by the QPU, and the two class probabilities produced by the trained readout.

When there are many quantum outcomes, the third panel shows the most likely photon patterns and combines the rest as **other**. Those bars still sum to one. The fourth panel always has two outputs: the model's probability for digit 0 and its probability for digit 1.

In [ ]:
# Supporting code: plots one image's complete path to classification. You can just run this cell.
example_index = 0

# Recreate the phase values fitted by the classifier; no QPU job is submitted here.
pca_training_values = classifier.reduction.transform(X_train)
pca_example = pca_training_values[example_index]
scaled_example = (pca_example - pca_training_values.min()) / (
    pca_training_values.max() - pca_training_values.min() + 1e-8
)
phase_settings = np.pi * scaled_example

# Read the cached, unstandardized QPU probabilities from fit_reservoir.
probabilities = classifier._fit_quantum_cache[example_index].numpy()
probabilities = probabilities / probabilities.sum()  # Display a normalized distribution.
shown_outcomes = min(22, len(probabilities))
most_likely = np.argsort(probabilities)[-shown_outcomes:][::-1]
distribution = probabilities[most_likely]
outcome_labels = [
    "|" + "".join(map(str, classifier.layer.output_keys[i])) + "⟩"
    for i in most_likely
]
if shown_outcomes < len(probabilities):
    distribution = np.append(distribution, max(0.0, 1.0 - distribution.sum()))
    outcome_labels.append("other")

# Apply the trained classical readout and convert its two scores to probabilities.
with torch.no_grad():
    class_scores = classifier(reservoir_features[example_index:example_index + 1])
    class_probabilities = torch.softmax(class_scores, dim=1)[0].numpy()
predicted_class = int(class_probabilities.argmax())
is_correct = predicted_class == int(y_train[example_index])

fig, axes = plt.subplots(1, 4, figsize=(19, 4))
axes[0].imshow(X_train[example_index].reshape(28, 28), cmap="gray", vmin=0, vmax=1)
axes[0].set_title(f"1. Input image: digit {CLASSES[y_train[example_index]]}")
axes[0].axis("off")

axes[1].bar(np.arange(PCA_FEATURES), phase_settings, color=PRIMARY)
axes[1].set_title("2. Features encoded as phases")
axes[1].set_xlabel("Phase shifter")
axes[1].set_ylabel("Phase (radians)")

axes[2].bar(np.arange(len(distribution)), distribution, color=PRIMARY)
axes[2].set_xticks(np.arange(len(distribution)), outcome_labels, rotation=60, ha="right", fontsize=7)
axes[2].set_title("3. QPU probability distribution")
axes[2].set_xlabel("Measured photon pattern")
axes[2].set_ylabel("Probability")
axes[2].set_ylim(0, max(distribution) * 1.15)

result_colour = PRIMARY if is_correct else ACCENT
class_colours = [PRIMARY, PRIMARY]
class_colours[predicted_class] = result_colour
axes[3].bar(["digit 0", "digit 1"], class_probabilities, color=class_colours)
axes[3].set_title(f"4. Prediction: {CLASSES[predicted_class]} ({'correct' if is_correct else 'incorrect'})")
axes[3].set_ylabel("Class probability")
axes[3].set_ylim(0, 1)

fig.suptitle("One image: pixels → optical phases → QPU probabilities → classification", color=FOREGROUND)
fig.tight_layout()
plt.show()

## 9. Predict unseen images with fresh QPU measurements

`predict(..., processor=qpu)` applies the fitted PCA locally, embeds each test image in the circuit's phase shifters, measures a fresh probability distribution on the same fixed QPU reservoir, and applies the trained readout locally.

The returned tensor contains one score for digit 0 and one for digit 1. The larger score becomes the prediction. With such a small, noisy experiment, accuracy can vary—the objective is to experience and understand the complete QPU workflow.

In [ ]:
# Time this second QPU call separately from training-feature extraction.
start = time.perf_counter()

# For each unseen image, repeat the same three-stage path used in training:
# local PCA → QPU probability distribution → trained local readout.
with torch.no_grad():
    test_scores = classifier.predict(X_test, processor=qpu)
    predictions = test_scores.argmax(dim=1).numpy()

elapsed = time.perf_counter() - start
correct = int((predictions == y_test).sum())
print(f"QPU prediction finished in {elapsed / 60:.1f} minutes")
print(f"Accuracy: {correct}/{len(y_test)} = {100 * correct / len(y_test):.1f}%")

In [ ]:
# Supporting code: displays predictions and a confusion matrix. You can just run this cell.
display_count = min(len(X_test), 12)
columns = min(4, display_count)
rows = math.ceil(display_count / columns)
fig, axes = plt.subplots(rows, columns, figsize=(2.3 * columns, 2.5 * rows), squeeze=False)
for ax, image, true_label, predicted_label in zip(
    axes.flat, X_test[:display_count], y_test[:display_count], predictions[:display_count]
):
    result = "correct" if predicted_label == true_label else "incorrect"
    ax.imshow(image.reshape(28, 28), cmap="gray", vmin=0, vmax=1)
    ax.set_title(
        f"true: {CLASSES[int(true_label)]}\npredicted: {CLASSES[int(predicted_label)]} ({result})",
        color=PRIMARY if result == "correct" else ACCENT, fontsize=9,
    )
    ax.axis("off")
for ax in axes.flat[display_count:]:
    ax.axis("off")
fig.suptitle("Predictions made from fresh QPU reservoir features", color=FOREGROUND)
fig.tight_layout()
plt.show()

ConfusionMatrixDisplay.from_predictions(
    y_test, predictions, display_labels=CLASSES, colorbar=False, cmap=RESULT_CMAP
)
plt.title("QPU test predictions")
plt.show()

## What actually happened on the quantum computer?

For every training and test image, your classical computer calculated PCA values and converted them into phase settings. The QPU then:

1. prepared `N_PHOTONS` photons across the reservoir's optical modes;
2. sent them through a fixed interferometer;
3. applied the image-dependent phase shifts;
4. sent the photons through the second fixed interferometer; and
5. measured which collision-free output pattern occurred, repeatedly.

Those measured probabilities—not a simulated substitute—became the quantum reservoir features. The classical readout learned from the training distributions and labels, then interpreted newly measured test distributions.

## Try your own experiment

Return to the settings cell and change one value at a time. The notebook will validate the configuration and all figures will resize automatically. Good next steps are:

- increase `TRAIN_IMAGES_PER_CLASS` and see whether predictions become more stable;
- increase `TEST_IMAGES_PER_CLASS` for a more informative accuracy estimate;
- change `PCA_FEATURES` and observe how the mode count and number of outcomes change;
- change `N_PHOTONS` and compare the resulting Fock-space sizes; or
- switch between `qpu:ascella` and `qpu:belenos` to explore their different limits.

Larger settings mean more QPU work and potentially longer queue and execution times.

# Conclusion

You have completed an end-to-end hybrid quantum classification experiment. A classical PCA compressed each image, a fixed photonic QPU reservoir transformed it into measurement features, and a trainable classical readout learned to predict the digit.

The central MerLin workflow is:

`ReservoirClassifier(...)` → `fit_reservoir(..., processor=qpu)` → `make_dataset(...)` → train the readout → `predict(..., processor=qpu)`

You can now change the dataset size, encoded feature count, photon count, or QPU from one settings cell while keeping the same workflow.